# 🏙️ Ambient Crowd & World Chatter MVP — Qwen3-TTS

This notebook generates ambient crowd chatter and NPC barks for various scenes using the **Qwen3-TTS CustomVoice** model. We define scenes, assign lines to preset speakers with specific instructions, and then assemble them into looping ambient audio beds with random silence intervals.

In [ ]:
!pip install -q qwen-tts soundfile numpy

import os
import gc
import torch
import random
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/ambient_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Qwen3-TTS CustomVoice model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
cv_model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)

SCENES = {
    "medieval_market": [
        ("Ryan", "a cheerful merchant hawking wares", "Fresh bread! Hot from the oven! Best in the whole market!"),
        ("Vivian", "a bored noblewoman complaining", "This heat is absolutely unbearable. Someone fetch me water immediately."),
        ("Ryan", "an excited child tugging at sleeve", "Mum! Mum! Can we get one? Can we? Please?"),
        ("Vivian", "a tired merchant at end of day", "Last of the apples! Half price! I'm not carrying them home!"),
        ("Ryan", "a town crier announcing news", "Hear ye! Hear ye! The festival begins at sundown! All citizens welcome!"),
        ("Vivian", "two women gossiping in hushed voices", "Did you hear about the blacksmith? Three wives! I'm just saying."),
        ("Ryan", "a drunk beggar muttering", "Spare a coin, friend... even the gods were generous once..."),
        ("Vivian", "a guard walking patrol", "Move along, nothing to see here. I said MOVE along."),
    ],
    "tavern_night": [
        ("Ryan", "a rowdy drunk singing off key", "And the baron fell INTO the mudpit! And THAT is why I no longer work at the castle!"),
        ("Vivian", "a barmaid shouting across the room", "Last call! Anyone who's still sober clearly isn't trying hard enough!"),
        ("Ryan", "two gamblers arguing over cards", "That's not a six, that's clearly a NINE! You've been cheating all night!"),
        ("Vivian", "a bard starting a new song", "Ladies and gentlemen! A tale of love, betrayal, and a very unfortunate goat!"),
        ("Ryan", "an old veteran telling war stories", "I've told this story fifty times and it gets better every time, I'm not ashamed."),
        ("Vivian", "someone laughing at a bad joke", "Oh that's terrible. That's the worst joke I've ever heard. Tell it again."),
        ("Ryan", "a quiet man asking for directions", "Excuse me... sorry... the road to Millhaven... is it the left fork or the right?"),
    ],
    "battlefield_distant": [
        ("Ryan", "a commanding officer shouting orders", "HOLD THE LINE! Nobody retreats! HOLD!"),
        ("Vivian", "a soldier calling for a medic", "MEDIC! We need a medic on the eastern flank! NOW!"),
        ("Ryan", "a soldier rallying troops", "For the kingdom! FOR THE PEOPLE! CHARGE!"),
        ("Vivian", "a wounded soldier being carried", "I'm fine... I'm fine... just... get me to the healer..."),
        ("Ryan", "a scout reporting in", "Enemy flanking left! Three hundred strong! We're being surrounded!"),
        ("Vivian", "a veteran soldier stoic in battle", "Keep your head down and keep moving. The dying is for later."),
    ],
    "dungeon_prisoners": [
        ("Ryan", "a prisoner tapping on cell wall", "Hey. Hey! Anyone out there? How long have you been in here?"),
        ("Vivian", "a prisoner muttering to herself", "...on the seventeenth day the guards changed shift... if I could just time it right..."),
        ("Ryan", "a broken prisoner sobbing quietly", "I had a family... I had a life... what did I do to deserve this place..."),
        ("Vivian", "an angry prisoner rattling chains", "When I get out of here — and I WILL get out — heads will ROLL!"),
        ("Ryan", "a mysterious prisoner", "They said the same thing about the last three who came through that door. None of them left."),
    ],
    "scifi_space_station": [
        ("Ryan", "a bored docking controller", "Bay seven is clear for landing. Please submit your manifests within the hour. Customs is backed up."),
        ("Vivian", "a frantic engineer on comms", "I need a plasma relay to deck four IMMEDIATELY. No, I cannot wait. The reactor is doing a THING."),
        ("Ryan", "a space trader haggling", "Listen, I'll give you the spice OR the navigation data. Not both. Pick one."),
        ("Vivian", "an automated announcement system", "Attention all personnel. Sector seven is experiencing zero gravity fluctuations. Hold on to something."),
        ("Ryan", "two crew members gossiping", "Did you see the new commander? Apparently she's never lost a ship. I give it two weeks out here."),
        ("Vivian", "a confused tourist", "Excuse me, which way to the observation deck? I've been walking in circles for an hour."),
    ]
}

In [ ]:
print("Generating individual scene clips...")

scene_audio_data = {scene: [] for scene in SCENES}
target_sr = 24000

for scene_name, lines in SCENES.items():
    print(f"\n--- Generating Scene: {scene_name} ---")
    for idx, (speaker, instruct, line) in enumerate(lines):
        try:
            audio, sr = cv_model.generate_custom_voice(
                text=line,
                language="English",
                speaker=speaker,
                instruct=instruct
            )
            
            # Flatten the audio and ensure target sample rate
            audio_np = audio.squeeze().cpu().numpy()
            
            filename = f"ambient_{scene_name}_{idx:02d}.wav"
            filepath = os.path.join(OUTPUT_DIR, filename)
            sf.write(filepath, audio_np, sr)
            
            scene_audio_data[scene_name].append(audio_np)
            target_sr = sr
            
            print(f"[{scene_name}] {speaker} ({instruct}): {line}")
            display(Audio(filepath))
            
        except Exception as e:
            print(f"Error generating {speaker} ({instruct}): {e}")

In [ ]:
print("Assembling ambient beds with random silence...")
assembled_beds = {}

for scene_name, audio_clips in scene_audio_data.items():
    if not audio_clips:
        continue
        
    bed_parts = []
    for i, clip in enumerate(audio_clips):
        bed_parts.append(clip)
        
        # Add random silence between 0.3s and 1.2s
        if i < len(audio_clips) - 1:
            silence_dur = random.uniform(0.3, 1.2)
            silence_samples = int(target_sr * silence_dur)
            bed_parts.append(np.zeros(silence_samples, dtype=np.float32))
            
    assembled_bed = np.concatenate(bed_parts)
    
    bed_filename = f"ambient_{scene_name}_bed.wav"
    bed_filepath = os.path.join(OUTPUT_DIR, bed_filename)
    sf.write(bed_filepath, assembled_bed, target_sr)
    assembled_beds[scene_name] = bed_filepath
    
    print(f"Assembled {scene_name} bed:")
    display(Audio(bed_filepath))

In [ ]:
print("=== Scene Comparison ===")
for scene_name, path in assembled_beds.items():
    print(f"{scene_name} bed:")
    display(Audio(path))

## 🔄 Looping Tips for Game Engines

The generated audio clips are standard **24kHz mono WAV** files, perfect for game engine import.

### Unity
1. Import the `ambient_*_bed.wav` file into your project.
2. On the Audio Clip import settings, you can check **Force To Mono** if needed (they should already be mono).
3. Add an **AudioSource** component to a GameObject in your scene.
4. Assign the clip and check **Loop**.
5. Use an **Audio Mixer** to control volume and add spatial blending (e.g., reverb for a dungeon).

### Godot
1. Drop the files into your FileSystem dock.
2. In the Import dock for the WAV file, set **Loop Mode** to Forward and reimport.
3. Use an **AudioStreamPlayer** (or `AudioStreamPlayer2D` / `3D`) and enable Autoplay if desired.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/ambient_audio", 'zip', OUTPUT_DIR)
files.download("/content/ambient_audio.zip")